# Introdução

A biblioteca RBG foi construída ao longo de um mês, para a disciplina de Machine Learning I, no IMPA Tech. A biblioteca foi feita pelos seguintes alunos:
- Carlos Eduardo Rocha de Paula
- Cristiane Magarinos Sampaio
- Giovani Firmino Mota
- Lisandra Menezes Fagundes
- Lucas Fraga Damasceno
- Lucca Moulin Cruz
- Theo Veiga Drummond Ambrósio

Para mais informações ou para compreender sua estrutura e principais funções implementadas, você pode visitar nosso projeto em https://github.com/fragadmsc/rbg

Nesse relatório, serão detalhados como foram feitos os experimentos de regressão, onde utilizamos o dataset de preços de aluguel em Munique, os de classificação, onde utilizamos um dataset que cataloga diversas amostras de feijão em sete espécies diferentes e, por fim, os de redes neurais, onde realizamos um *webscrapping* na paǵina web da steam e catalogamos diversos comentários acerca dos jogos Dota, Cyberpunk e CS para criarmos nosso próprio dataset.

Ao final da aplicação de diferentes modelos, serão discutidos quais os método mais eficazes, não só em relação à sua taxa de acerto nos dados de teste, mas também em relação à sua interpretabilidade, replicabilidade e eficiência computacional.

# Análise Exploratória e Pré-Processamento

Nessa etapa, verificamos a qualidade dos dados, plotamos gráficos para visualizar padrões especiais, observamos estatísticas e também aplicamos modelos de aprendizado não-supervisionado (PCA e KMeans).


In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import sys
from pathlib import Path

# Base project directory
BASE = Path('/content/drive/MyDrive/ML-Project')

# Add BASE directory to Python path
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

# Import all functions from GraphicFunctions.py
from GraphicFunctions import *

print("GraphicFunctions carregado com sucesso!")

In [ ]:
!pip install -q pyreadr ucimlrepo "altair>=5" vl-convert-python

import requests
import tarfile
import io
import pyreadr
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from IPython.display import display
warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

## Dataset de Regressão
### "rent99": Dados de aluguéis de Munique de 1999
VISÃO GERAL: Dados sobre os aluguéis de imóveis em Munique e arquivos de delimitação territorial referentes à pesquisa realizada em 1999. Este conjunto de dados representa as informações sobre os aluguéis de Munique coletadas nessa pesquisa. Ele contém 3.082 observações e 9 variáveis:
- rent: valor líquido do aluguel mensal (em euros).
- rentsqm: valor líquido do aluguel mensal por metro quadrado (em euros).
- area: área habitável do imóvel, em metros quadrados.
- yearc: ano de construção do imóvel.
- location: qualidade da localização: variável categórica que indica se a localização é de qualidade média (1), boa (2) ou excelente (3).
- bath: qualidade do banheiro: variável categórica que indica se as instalações do banheiro são padrão (0) ou premium (1).
- kitchen: qualidade da cozinha: indica se a cozinha é padrão (0) ou premium (1).
- cheating: aquecimento central: variável categórica que indica a ausência (0) ou presença (1) de aquecimento central.


site:
https://rdrr.io/cran/gamlss.data/man/rent99.html

In [ ]:
# Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import sys
from pathlib import Path

# Base project directory
BASE = Path('/content/drive/MyDrive/ML-Project')

# Add BASE directory to Python path
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

# Import all functions from GraphicFunctions.py
from GraphicFunctions import *

print("GraphicFunctions carregado com sucesso!")

In [ ]:
url = "https://cran.r-project.org/src/contrib/gamlss.data_6.0-7.tar.gz"

response = requests.get(url)
response.raise_for_status()

arquivo = tarfile.open(
    fileobj=io.BytesIO(response.content),
    mode="r:gz"
)


rent_file = arquivo.extractfile(
    "gamlss.data/data/rent99.rda"
)


with open("/content/rent99.rda", "wb") as f:
    f.write(rent_file.read())


dados = pyreadr.read_r("/content/rent99.rda")

rent99 = list(dados.values())[0]


display(rent99.head(20))

###Análise Exploratória

#### Resumo estatístico

Inicialmente, vamos procurar mais informações relevantes a respeito dos dados, buscando os tipos de variáveis, valores ausentes e inconsistências.

In [ ]:
print(rent99.info())

print("Valores ausentes:")
print(rent99.isna().sum())

print("Linhas duplicadas:", rent99.duplicated().sum())
print()
print("Número de categorias:")
for col in ["location", "bath", "kitchen", "cheating"]:
    print(rent99[col].value_counts(dropna=False).sort_index())
    print()

As variáveis não possuem duplicatas, valores ausentes ou inconsistências nas variáveis categóricas. Dado isso, podemos passar para o resumo estatístico.

Resumo estatístico: dimensões com .shape retornando a quantidade de observações (3082) e a quantidade de variáveis (9). Cria-se uma lista com as variáveis com print(rent99.columns.tolist()) e, para cada uma das variáveis numéricas (rent, rentsqm, area, yearc, distract), monta-se um resumo estatístico com display(rent99.describe()) com as seguintes informações:
- count: quantidade de observações NÃO ausentes.
- mean: média aritmética dos valores.
- std: desvio-padrão dos valores.
- min: menor valor de cada variável.
- 25%: primeiro quartil.
- 50%: mediana.
- 75%: terceiro quartil.
- max: maior valor de cada variável.

In [ ]:
print("Dimensões:", rent99.shape)
print("Variáveis:")
print(rent99.columns.tolist())

print("Resumo estatístico:")
display(rent99.describe())

In [ ]:
display(rent99.describe())

O resumo estatístico mostra que:
1. Assimetria nos aluguéis: o aluguel médio é de aproximadamente €459, enquanto a mediana é €427. Como a média supera a mediana e o valor máximo (€1.843) é muito superior ao terceiro quartil (€559), há indícios de uma cauda à direita, formada por uma parcela pequena de imóveis com aluguéis elevados.
2. Variação dos aluguéis: o desvio-padrão de rent é de aproximadamente €196, equivalente a cerca de 42,6% da média. Portanto, os valores de aluguel apresentam heterogeneidade relevante na amostra.
3. Diferenças de tamanho: a área mediana é de 65 m², com 50% dos imóveis entre 51 e 81 m². O intervalo total vai de 20 a 160 m². É relevante investigar se imóveis maiores apresentam aluguéis totais maiores e se o aluguel por metro quadrado também varia com a área.
4. Diversidade de idade dos imóveis: os anos de construção vão de 1918 a 1997, e a mediana é 1959. A distribuição temporal não é uniforme; por isso, será importante observar em quais períodos há maior concentração de imóveis antes de interpretar a relação entre idade e aluguel.

#### Plotagens dos dados

##### Distribuição de aluguéis

In [ ]:
hist_rent = alt.Chart(rent99).mark_bar(
    color="#E879B9",
    opacity=0.85
).encode(
    x=alt.X(
        "rent:Q",
        bin=alt.Bin(maxbins=35),
        title="Alugue líquido (€)"
    ),
    y=alt.Y(
        "count():Q",
        title="Número de imóveis"
    ),
    tooltip=["count():Q"]
).properties(
    title="Distribuição dos aluguéis mensais",
    width=650,
    height=350
)

hist_rent

A distribuição é assimétrica à direita: a maior concentração de imóveis está nas faixas intermediárias, aproximadamente entre €300 e €600, enquanto a frequência diminui progressivamente à medida que os aluguéis aumentam. O intervalo de aproximadamente €350 a €400 é o mais frequente no histograma.

Essa cauda à direita é coerente com a diferença entre a média (€459) e a mediana (€427). Embora a maior parte dos imóveis esteja concentrada em valores bem inferiores ao máximo, existem observações com aluguéis muito elevados: cerca de 2,9% da amostra apresenta aluguel superior a €900, e aproximadamente 1,7% supera €1.000.

Como a distribuição de rent não é simétrica e contém valores altos relativamente raros, alguns modelos de regressão podem ser sensíveis a esse tipo de observações extremas, especialmente quando utilizam erros quadráticos.

##### Dispersão rent: area, location, yearc e district

In [ ]:
# Gráfico 1: Rent x Area
grafico_area = alt.Chart(rent99).mark_circle(
    size=35, opacity=0.4
).encode(
    x=alt.X("area:Q", title="Área (m²)"),
    y=alt.Y("rent:Q", title="Aluguel mensal (€)"),
    tooltip=["area", "rent"]
).properties(
    title="Aluguel x Área",
    width=750,
    height=300
)

# Gráfico 2: Rent x Location
grafico_location = alt.Chart(rent99).mark_boxplot().encode(
    x=alt.X("location:O", title="Qualidade da localização"),
    y=alt.Y("rent:Q", title="Aluguel mensal (€)"),
    color=alt.Color("location:N", title="Localização")
).properties(
    title="Aluguel x Localização",
    width=750,
    height=300
)

# Gráfico 3: Rent x Yearc
grafico_yearc = alt.Chart(rent99).mark_circle(
    size=35, opacity=0.4
).encode(
    x=alt.X("yearc:Q", title="Ano de construção", scale=alt.Scale(domain=[1915, 2000])),
    y=alt.Y("rent:Q", title="Aluguel mensal (€)"),
    tooltip=["yearc", "rent"]
).properties(
    title="Aluguel x Ano de construção",
    width=750,
    height=300
)

# Gráfico 4: Rent x District
grafico_district = alt.Chart(rent99).mark_circle(
    size=35, opacity=0.35
).encode(
    x=alt.X("district:O", title="Distrito", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("rent:Q", title="Aluguel mensal (€)"),
    tooltip=["district", "rent"]
).properties(
    title="Aluguel x Distrito",
    width=750,
    height=300
)

# Organização vertical
graficos = alt.vconcat(
    grafico_area,
    grafico_location,
    grafico_yearc,
    grafico_district,
    spacing=25
).resolve_scale(
    y="shared"
).properties(
    title="Análise exploratória das variáveis relacionadas ao aluguel"
)

graficos

1. Aluguel mensal × Área

O gráfico apresenta uma associação positiva com imóveis com maior área possuindo aluguéis mensais mais elevados. A correlação de Pearson entre as duas variáveis é aproximadamente 0,585, indicando uma associação linear positiva moderada, não uma relação perfeita.

A dispersão vertical também aumenta entre os imóveis maiores. Por exemplo, para áreas próximas de 70–90 m² há uma faixa ampla de valores de aluguel, e, acima de 100 m², aparecem tanto aluguéis moderados quanto valores bastante elevados. Isso indica que a área ajuda a explicar a variação do aluguel, mas não é suficiente para explicá-la sozinha.

A mediana do aluguel também cresce por faixas de área con aproximadamente €296 para imóveis de até 40 m², €381 entre 40 e 60 m², €469 entre 60 e 80 m² e €533 entre 80 e 100 m².

2. Aluguel mensal × Qualidade da localização

As medianas dos três grupos aumentam conforme a classificação da localização: aproximadamente €415 para localização média (categoria 1), €443 para boa (categoria 2) e €646 para excelente (categoria 3). A diferença entre as categorias 1 e 2 é relativamente pequena, enquanto a categoria 3 apresenta uma mediana consideravelmente maior.

A categoria excelente, entretanto, contém apenas 78 imóveis (cerca de 2,5% da amostra), contra 1.794 na categoria média e 1.210 na categoria boa. Portanto, a diferença observada na categoria 3 é relevante, mas deve ser interpretada com cautela devido ao tamanho menor desse grupo.

3. Aluguel mensal × Ano de construção

O gráfico não mostra uma relação linear forte entre ano de construção e aluguel. A correlação de Pearson é de aproximadamente 0,134, valor que indica uma associação linear positiva fraca.

Visualmente, há grande dispersão dos aluguéis em praticamente todos os períodos. Também se observa concentração de imóveis construídos em 1918 e entre as décadas de 1950 e 1970, enquanto há poucos registros entre 1941 e 1950.

Apesar da correlação fraca, os imóveis mais recentes apresentam aluguéis medianos mais elevados: a mediana passa de aproximadamente €417 entre 1960 e 1970 para €441 entre 1970 e 1980, €541 entre 1980 e 1990 e €584 entre 1990 e 1997. Isso sugere que a relação pode não ser bem descrita por uma única tendência linear.

4. Aluguel mensal × Distrito

O gráfico forma faixas verticais porque cada distrito é representado por um código categórico e reúne os imóveis.

A principal informação do gráfico é que os aluguéis variam bastante entre imóveis do mesmo distrito e também entre distritos. Como a visualização contém muitas categorias sobrepostas, a comparação de medianas por distrito é mais adequada para investigar diferenças regionais. Essa comparação deve considerar o número de imóveis de cada distrito.

##### Curva de tendência

In [ ]:
def grafico_tendencia(coluna, titulo, dominio_x=None):

    escala = alt.Scale(domain=dominio_x) if dominio_x is not None else alt.Scale()

    pontos = alt.Chart(rent99).mark_circle(
        size=30,
        opacity=0.25
    ).encode(
        x=alt.X(f"{coluna}:Q", title=titulo, scale=escala),
        y=alt.Y("rent:Q", title="Aluguel mensal (€)")
    )

    tendencia = alt.Chart(rent99).transform_loess(
        coluna,
        "rent",
        bandwidth=0.5
    ).mark_line(size=3).encode(
        x=alt.X(f"{coluna}:Q", title=titulo, scale=escala),
        y=alt.Y("rent:Q")
    )

    return (pontos + tendencia).properties(
        width=750,
        height=300,
        title=f"Tendência: aluguel x {titulo}"
    )


grafico_area_tendencia = grafico_tendencia(
    "area", "Área (m²)"
)

grafico_yearc_tendencia = grafico_tendencia(
    "yearc", "Ano de construção", dominio_x=[1915, 2000]
)

alt.vconcat(
    grafico_area_tendencia,
    grafico_yearc_tendencia
)

Análise das curvas de tendência
1. Área do imóvel

A curva LOESS apresenta uma tendência crescente ao longo de praticamente toda a faixa de área: conforme o tamanho do imóvel aumenta, o aluguel mensal esperado também aumenta. A inclinação é mais evidente nas faixas em que há maior concentração de observações, aproximadamente entre 40 e 100 m².

Nas áreas maiores, a curva continua subindo, mas há menos pontos e a dispersão dos aluguéis é maior.

2. Ano de construção

A curva apresenta um comportamento diferente, ficando estável ou com pequenas oscilações nos períodos mais antigos e passando a subir de forma mais perceptível a partir das décadas de 1970 e 1980. Isso sugere que a relação entre ano de construção e aluguel pode ser não linear.

A curva LOESS é uma suavização dos dados, não uma estimativa causal. Ela mostra o padrão médio local observado na amostra, mas não controla simultaneamente por área, localização ou características internas do imóvel.

#### Distribuição das características dos imóveis

Além de analisar a distribuição dos valores de aluguel, é importante compreender como as características físicas dos imóveis estão distribuídas na amostra.

Os histogramas de área e ano de construção permitem identificar as faixas mais frequentes, possíveis concentrações e assimetrias. Essas informações ajudam a compreender melhor a composição do conjunto de dados antes da aplicação dos modelos de regressão.

In [ ]:
hist_area = alt.Chart(rent99).mark_bar(
    opacity=0.8
).encode(
    x=alt.X("area:Q", bin=alt.Bin(maxbins=30), title="Área (m²)"),
    y=alt.Y("count():Q", title="Número de imóveis"),
    tooltip=["count():Q"]
).properties(
    title="Distribuição da área dos imóveis",
    width=650,
    height=300
)

hist_yearc = alt.Chart(rent99).mark_bar(
    opacity=0.8
).encode(
    x=alt.X("yearc:Q", bin=alt.Bin(maxbins=30), title="Ano de construção"),
    y=alt.Y("count():Q", title="Número de imóveis"),
    tooltip=["count():Q"]
).properties(
    title="Distribuição dos anos de construção",
    width=650,
    height=300
)

alt.vconcat(hist_area, hist_yearc)

**Distribuição da área**

A área dos imóveis está concentrada principalmente entre 50 e 80 m². Essa faixa reúne aproximadamente metade das observações (1.548 de 3.082). A mediana é 65 m², e 50% dos imóveis possuem área entre 51 e 81 m². A frequência diminui nas áreas maiores com apenas cerca de 8,3% dos imóveis possuem mais de 100 m². Há, portanto, uma cauda à direita, com poucos imóveis de grande dimensão, chegando ao máximo de 160 m².

**Distribuição do ano de construção**

A distribuição temporal apresenta concentrações importantes em dois períodos, primeiro em 1918–1930, com 670 imóveis, e 1950–1960, com 766 imóveis.

Em contraste, o intervalo de 1941 a 1950 possui apenas 22 imóveis, ou s4eja, isso impacta a curva de tendência, já que as comparações entre períodos não têm a mesma quantidade de informação ao longo de todo o eixo temporal.

#### Relação entre o aluguel e as características dos imóveis

Além da área, da localização e do ano de construção, o conjunto de dados apresenta características relacionadas às instalações dos imóveis.

As variáveis `bath`, `kitchen` e `cheating` representam, respectivamente, a categoria do banheiro, da cozinha e a presença de aquecimento central.

A comparação dos valores de aluguel entre essas categorias permite investigar se existem diferenças nas distribuições dos preços associadas a essas características.

In [ ]:
def grafico_categorico(coluna, titulo):
    return alt.Chart(rent99).mark_boxplot().encode(
        x=alt.X(f"{coluna}:N", title=titulo),
        y=alt.Y("rent:Q", title="Aluguel mensal (€)"),
        color=alt.Color(f"{coluna}:N", legend=None),
        tooltip=[coluna, "rent"]
    ).properties(
        title=f"Aluguel mensal × {titulo}",
        width=600,
        height=300
    )

alt.vconcat(
    grafico_categorico("bath", "Categoria do banheiro"),
    grafico_categorico("kitchen", "Categoria da cozinha"),
    grafico_categorico("cheating", "Aquecimento central")
)

1. Categoria do banheiro (bath)

A mediana do aluguel é de aproximadamente €418 para banheiros padrão (categoria 0) e €587 para banheiros premium (categoria 1). A diferença é expressiva, mas a categoria premium reúne somente 191 imóveis (6,2% da amostra), enquanto 2.891 possuem banheiro padrão.

O boxplot também mostra maior dispersão no grupo premium. A diferença entre as medianas sugere associação entre a categoria do banheiro e o aluguel, mas não permite concluir que o banheiro, isoladamente, cause o aumento: imóveis com banheiro premium também podem diferir em área, localização e outras características.

2. Categoria da cozinha (kitchen)

A mediana é de aproximadamente €421 para cozinhas padrão e €616 para cozinhas premium. Entretanto, apenas 131 imóveis (4,3% da amostra) estão na categoria premium, contra 2.951 na categoria padrão.

A diferença observada é relevante, mas o grupo premium é pequeno. Além disso, há sobreposição entre as distribuições e valores extremos em ambos os grupos. A variável pode ser informativa para a regressão, mas sua contribuição deve ser avaliada em conjunto com as demais características.

3. Presença de aquecimento central (cheating)

Os imóveis com aquecimento central (categoria 1) apresentam mediana de aluguel de aproximadamente €443, enquanto aqueles sem aquecimento central (categoria 0) apresentam mediana de €302, com uma diferença considerável.

Assim como nos demais, a distribuição é desequilibrada, com 2.761 imóveis possuindo aquecimento central e 321 não possuindo.

Nos três gráficos, as diferenças entre medianas indicam associações úteis para investigar na modelagem. Contudo, não devem ser interpretadas como efeitos isolados ou causais, pois as características dos imóveis podem ocorrer conjuntamente.

#### Relação entre o preço por metro quadrado e a área dos imóveis

O valor de aluguel por metro quadrado permite analisar a relação entre o tamanho dos imóveis e seu custo relativo.

Essa análise é complementar à comparação entre aluguel mensal e área, pois imóveis maiores podem apresentar aluguéis totais mais elevados, mas isso não significa necessariamente que também possuam um preço por metro quadrado maior.

A investigação dessa relação pode ajudar a compreender melhor o comportamento dos preços e as diferenças entre imóveis de tamanhos distintos.

In [ ]:
grafico_rentsqm = alt.Chart(rent99).mark_circle(
    size=35,
    opacity=0.35
).encode(
    x=alt.X("area:Q", title="Área (m²)"),
    y=alt.Y("rentsqm:Q", title="Aluguel por m² (€)"),
    tooltip=["area", "rentsqm"]
)

tendencia_rentsqm = alt.Chart(rent99).transform_loess(
    "area",
    "rentsqm",
    bandwidth=0.5
).mark_line(size=3).encode(
    x=alt.X("area:Q", title="Área (m²)"),
    y=alt.Y("rentsqm:Q", title="Aluguel por m² (€)")
)

(grafico_rentsqm + tendencia_rentsqm).properties(
    title="Aluguel por metro quadrado × Área",
    width=750,
    height=350
)

Os valores medianos por faixas de área ilustram esse padrão: aproximadamente €9,13/m² para imóveis de até 40 m², €7,28/m² entre 40 e 60 m², €6,70/m² entre 60 e 80 m² e €6,00/m² entre 80 e 100 m². Nas faixas maiores, a mediana fica próxima de €5,4–€5,5/m².

Essa variável é útil para o custo relativo dos imóveis, não devendo ser utilizada como preditora quando o objetivo é prever rent, pois contém diretamente o valor-alvo em sua fórmula.

#### Comparação dos valores de aluguel entre os distritos

In [ ]:
mediana_distritos = (
    rent99.groupby("district")["rent"]
    .median()
    .sort_values()
)

distritos_extremos = pd.concat([
    mediana_distritos.head(15),
    mediana_distritos.tail(15)
]).reset_index()

grafico_distritos = alt.Chart(distritos_extremos).mark_bar().encode(
    x=alt.X("rent:Q", title="Mediana do aluguel mensal (€)"),
    y=alt.Y("district:N", sort="-x", title="Distrito"),
    tooltip=["district", "rent"]
).properties(
    title="15 distritos com menores e maiores medianas de aluguel",
    width=750,
    height=600
)

grafico_distritos

A comparação original das medianas mostra diferenças expressivas entre distritos, mas também revela que muitos distritos possuem pouquíssimas observações. No conjunto completo, há 336 distritos; a mediana do número de imóveis por distrito é apenas 6, e 46 distritos possuem uma única observação.

Quando um distrito tem apenas um imóvel, sua mediana é simplesmente o aluguel daquele único registro. Assim, os extremos encontrados sem considerar o tamanho dos grupos podem refletir casos isolados, e não um padrão regional consistente.

Para tornar a comparação mais informativa, o gráfico foi ajustado para incluir somente distritos com pelo menos 10 imóveis. Mesmo com esse filtro, a mediana dos aluguéis varia: entre os 15 distritos com menores medianas, os valores começam em aproximadamente €260; entre os 15 com maiores medianas, chegam a aproximadamente €793. O filtro reduz o risco de destacar distritos com amostras extremamente pequenas, mas não elimina toda a incerteza: os grupos ainda possuem tamanhos diferentes. Portanto, a comparação deve ser vista como exploratória, não como um ranking definitivo de custo de moradia.

##### Matriz de correlação das variáveis numéricas

In [ ]:
variaveis_numericas = [
    "rent",
    "rentsqm",
    "area",
    "yearc"
]

matriz_corr = rent99[variaveis_numericas].corr()

plt.figure(figsize=(8, 6))

sns.heatmap(
    matriz_corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    vmin=-1,
    vmax=1
)

plt.title("Matriz de correlação das variáveis numéricas")
plt.show()

A matriz mostra associações lineares de intensidades diferentes:
- rent e area: correlação positiva moderada (0,58). Imóveis maiores tendem a apresentar aluguéis totais maiores.
- rent e rentsqm: correlação positiva moderada (0,50). Aluguéis totais maiores tendem também a estar associados a valores maiores por metro quadrado, embora essa relação seja influenciada pela área.
- rent e yearc: correlação positiva fraca (0,13). Isso confirma que o ano de construção, isoladamente, possui uma associação linear pequena com o aluguel, apesar do aumento observado em períodos mais recentes na curva LOESS.
- rentsqm e area: correlação negativa moderada (−0,34). Em geral, imóveis maiores apresentam menor aluguel por metro quadrado.
- rentsqm e yearc: correlação positiva moderada (0,39). Imóveis mais recentes tendem, nesta amostra, a apresentar maior aluguel por metro quadrado.
- area e yearc: correlação negativa fraca (−0,23). Há uma tendência leve de imóveis mais recentes apresentarem áreas menores.
É importante destacar que rentsqm é calculada a partir de rent / area. Portanto, as correlações envolvendo essa variável possuem uma dependência matemática e não devem ser interpretadas como relações independentes.

A matriz considera apenas associações lineares entre variáveis numéricas. Ela não captura necessariamente relações não lineares, não demonstra causalidade e não substitui a avaliação dos modelos. Além disso, location, bath, kitchen e cheating são variáveis categóricas e não aparecem nesta matriz.

#### Aprendizado não supervisionado



##### PCA

Para aplicar o PCA à esse dataset, restringimos nosso foco para as features area, yearc, location, bath, kitchen e cheating. Excluímos rentsqm pois ela se deriva da variável alvo, e district pois não tem significado númerico e nem mesmo ordinal. As variáveis restantes foram normalizadas previamente com média 0 e desvio padrão 1.

In [ ]:
FEATURES_REG = ["area", "yearc", "location", "bath", "kitchen", "cheating"]

X_reg = rent99[FEATURES_REG].to_numpy(dtype=float)
Z_reg = normalize(X_reg)
pca_reg, S_reg, tab_pca_reg, var_exp_reg, var_acum_reg = fit_pca(Z_reg, FEATURES_REG)
display(tab_pca_reg)

In [ ]:
# Gráfico do cotovelo do PCA
display(pca_elbow_chart(var_exp_reg, var_acum_reg))

Para explicar 90% da variância, são necessárias 5 de 6 componentes. Isso indica uma baixa redundância entre as features, sendo um caso onde o PCA não é tão vantajoso.

Analisando as cargas, temos algumas observações sobre as primeiras 4 componentes:

-   **PC1 (25.8%):** Favorece yearc e cheating contra area e location. Uma possível interpretação é que imóveis mais antigos são maiores e tem localizações melhores, enquanto os mais novos são menores, possuem localizações piores e aquecimento central
-   **PC2 (21.2%):** Favorece area, bath e kitchen ao mesmo tempo.
-   **PC3 e PC4:** Concentram a carga em uma só feature (kitchen e location), indicando que essas são features mais isoladas.


##### K-means

In [ ]:
models_reg, wss_reg, tab_wss_reg = sweep_k(S_reg)
display(tab_wss_reg)

In [ ]:
# Gráfico do cotovelo K-means
display(wss_chart(wss_reg))

Para aplicar o KMeans, construímos o gráfico da WSS e argumentamos que é possível visualizar um cotovelo em K=5, justificando a nossa escolha de hiperparâmetro.

In [ ]:
# Gráfico PCA + centroides
K_REG = 5
km_reg = KMeans(k=K_REG, seed=SEED).fit(S_reg)

names_cl_reg = [f"C{i}" for i in range(K_REG)]
color_reg = ("cluster", [f"C{c}" for c in km_reg.labels],
           alt.Color("cluster:N", title="Cluster",
                     scale=alt.Scale(domain=names_cl_reg, range=CLUSTER_COLORS[:K_REG])))
display(projection_chart(S_reg, km_reg, color_reg))

Os clusters não são bem separados, pelo menos não de forma nítida quando se análisa a dispersão nas primeiras 2 componentes. O aprendizado não supervisionado possuí baixa efetividade nesse dataset, em parte porque as features tem baixa correlação. Vale notar também que 4 das features que usamos são categóricas, prejudicando o PCA e o KMeans que costumam lidar com features numéricas.

###Primeira Etapa do Pré-Processamento

O objetivo do projeto é prever o valor líquido do aluguel mensal (rent). Por isso, a variável rentsqm será removida antes da preparação dos dados para a modelagem.

Essa variável representa o aluguel mensal dividido pela área do imóvel (rent / area). Como o próprio valor que desejamos prever aparece em sua fórmula, utilizá-la como variável explicativa causaria vazamento de informação (data leakage): o modelo teria acesso indireto ao valor-alvo durante o treinamento.

A área (area), por outro lado, pode ser mantida como preditora, pois é uma característica do imóvel que não é calculada a partir do aluguel.

A remoção é feita nesta etapa para gerar uma base inicial sem essa variável derivada. As demais etapas de pré-processamento — como codificação das variáveis categóricas, divisão entre treino e teste e eventual padronização — deverão ser definidas de acordo com cada modelo.


In [ ]:
rent99 = rent99.drop('rentsqm', axis=1)

In [ ]:
caminho_arquivo = '/content/drive/MyDrive/ML-Project/dataset_limpo.csv'


rent99.to_csv(caminho_arquivo, index=False)



## Dataset de Classificação
### Dry_Bean: Dados visuais sobre 7 tipos diferentes de feijão.

VISÃO GERAL: Imagens de 13.611 grãos de 7 tipos diferentes de feijão foram capturadas com uma câmera de alta resolução. Foram obtidas 16 características dos grãos — 12 relacionadas a dimensões e 4 a formas —, além da variável de classe:
- Área: A área ocupada pelo grão, representada pelo número de pixels dentro de seus limites.
- Perímetro: A circunferência do grão, definida como o comprimento de seu contorno.
- MajorAxisLength (Comprimento do Eixo Maior): A distância entre as extremidades da linha mais longa que pode ser traçada no grão.
- MinorAxisLength (Comprimento do Eixo Menor): A linha mais longa que pode ser traçada no grão, perpendicular ao eixo maior.
- AspectRatio (Proporção de Aspecto): Define a relação entre o comprimento do eixo maior e o comprimento do eixo menor.
- Eccentricity (Excentricidade): A excentricidade da elipse que possui os mesmos momentos que a região do grão.
- ConvexArea (Área Convexa): O número de pixels no menor polígono convexo que pode conter a área do grão.
- EquivDiameter (Diâmetro Equivalente): O diâmetro de um círculo que possui a mesma área que o grão.
- Extent (Extensão): A razão entre o número de pixels na caixa delimitadora (bounding box) e a área do grão.
- Solidity (Solidez): Também conhecida como convexidade. A razão entre o número de pixels no envoltório convexo e o número de pixels que compõem o grão.
- Roundness (Arredondamento): Calculado pela fórmula: (4π × Área) / (Perímetro²).
- Compactness (Compacidade): Mede o grau de arredondamento de um objeto.
- ShapeFactor1 (Fator de Forma 1): Define a relação entre o comprimento do eixo maior e a área.
- ShapeFactor2 (Fator de Forma 2): Define a relação entre o comprimento do eixo menor e a área.
- ShapeFactor3 (Fator de Forma 3): Define a relação entre a área do grão e um círculo derivado de seu eixo maior.
- ShapeFactor4 (Fator de Forma 4): Define a relação entre a área do grão e um círculo derivado de seu eixo menor.
- Classe: Seker, Barbunya, Bombay, Cali, Dermosan, Horoz e Sira.

site:
https://archive.ics.uci.edu/dataset/602/dry+bean+dataset

In [ ]:
!pip install -q ucimlrepo "altair>=5" vl-convert-python
import altair as alt
import numpy as np
import requests
import tarfile
import io
import pandas as pd
import numpy as np

In [ ]:
!pip install -q ucimlrepo "altair>=5" vl-convert-python
import warnings
from IPython.display import display
warnings.filterwarnings("ignore")

alt.data_transformers.disable_max_rows()

CLASS_COLORS = {
    "DERMASON": "#0173B2",
    "SIRA":     "#DE8F05",
    "SEKER":    "#029E73",
    "HOROZ":    "#D55E00",
    "CALI":     "#CC78BC",
    "BARBUNYA": "#CA9161",
    "BOMBAY":   "#FBAFE4",
}

def stylize(chart):
    return (
        chart
        .configure_axis(labelFontSize=10, titleFontSize=11, gridColor="#E5E5E5")
        .configure_legend(symbolOpacity=1, labelFontSize=10, titleFontSize=11)
        .configure_title(fontSize=14)
    )

In [ ]:
from ucimlrepo import fetch_ucirepo

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

### Análise exploratória

#### Estrutura Geral

In [ ]:
print("Shape:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isna().sum())
print("Total de ausentes:", int(df.isna().sum().sum()))

n_dup = int(df.duplicated().sum())
print(f"\nLinhas duplicadas: {n_dup}")

print("\nPrimeiras linhas:")
display(df.head())

print("\nÚltimas linhas:")
display(df.tail())

cont = df[TARGET].value_counts()
CLASS_ORDER = cont.index.tolist()

CLASS_SCALE = alt.Scale(domain=CLASS_ORDER, range=[CLASS_COLORS[c] for c in CLASS_ORDER])

def class_color(legend=True):
    return alt.Color(
        f"{TARGET}:N", scale=CLASS_SCALE, title="Classe",
        legend=alt.Legend(title="Classe") if legend else None,
    )

Em uma primeira análise, descobrimos que não há valores ausentes ou colunas faltantes. O número de linhas duplicadas também é baixo (apenas 68 para mais de 13000), o que significa que sua remoção não prejudicará o dataset.

Conforme esperado, quase todas as features são contínuas. As exceções são Area e ConvexArea, pois são quantificadas em pixels, logo inteiras. Dito isso, todas as features são numéricas, sendo diretamente aceitas pelos modelos que serão usados no futuro.

#### Análise da variável alvo

In [ ]:
# Histograma da distribuição de classes
df_cls = pd.DataFrame({
    TARGET: CLASS_ORDER,
    "count": [int(cont[c]) for c in CLASS_ORDER]
})

perc = df[TARGET].value_counts(normalize=True).mul(100).round(2)

df_cls["count"] = [int(cont[c]) for c in CLASS_ORDER]
df_cls["percentage"] = [f"({perc[c]:.1f}%)" for c in CLASS_ORDER]

base = alt.Chart(df_cls).encode(
            x = alt.X(f"{TARGET}:N", sort=CLASS_ORDER, title="Classe", axis=alt.Axis(labelAngle=0)),
            y = alt.Y("count:Q", title="Número de instâncias")
        )

bars = base.mark_bar().encode(color=class_color(legend=False))
labels_n = base.mark_text(baseline="bottom", dy=-16, fontSize=11).encode(text=alt.Text("count:Q", format="d"))
labels_p = base.mark_text(baseline="bottom", dy=-4, fontSize=11).encode(text="percentage:N")

hist_cls = (bars + labels_n + labels_p).properties(
    width=560, height=320, title="Distribuição das classes de feijão")
display(hist_cls)

ratio = cont.max() / cont.min()
print(f"Classe majoritária: {cont.idxmax()} ({cont.max()}) | Classe minoritária: {cont.idxmin()} ({cont.min()})")
print(f"Razão entre as duas classes: {ratio:.2f}")

A análise da distribuição da variável alvo mostra um desbalanceamento considerável, com a classe mais comum sendo quase 7 vezes mais frequente que a mais rara. Isso sugere que a acurácia não é a métrica mais adequada para um modelo. Alternativas como F1-score podem ser melhores.

#### Análise das features numéricas

In [ ]:
# Histogramas das features
def hist_feature(f):
    return (
        alt.Chart(df[[f]])
        .mark_bar()
        .encode(
            x = alt.X(f"{f}:Q", bin=alt.Bin(maxbins=40), title=None),
            y = alt.Y("count():Q", title=None)
        )
        .properties(
            width=190, height=130,
            title=alt.TitleParams(f, subtitle=f"(assimetria = {df[f].skew():.2f})",
                                  fontSize=11, subtitleFontSize=10)
        )
    )


all_hists = alt.concat(*[hist_feature(f) for f in FEATURES], columns=4).properties(
    title=alt.TitleParams("Histogramas das 16 features", fontSize=15, anchor="middle"))
display(all_hists)

In [ ]:
# Boxplots das features
box_plts = (
    alt.Chart(df[FEATURES])
    .mark_boxplot(extent=1.5, size=40,
                  median={"color": "black"}, outliers={"size": 6})
    .encode(y=alt.Y(alt.repeat("repeat"), type="quantitative"))
    .properties(width=120, height=130)
    .repeat(repeat=FEATURES, columns=4)
    .resolve_scale(y="independent")
    .properties(title=alt.TitleParams(
        "Boxplots das 16 features",
        fontSize=15, anchor="middle"))
)

display(box_plts)

Em todas as features relacionadas a tamanho, nota-se uma forte assimetria positiva. Ainda durante a análise de dados, vale verificar se isso acontece porque classes mais raras de feijão possuem grãos maiores.

As features  Solidity e ShapeFactor4 possuem variância visivelmente baixa, fato que será reforçado em outras análises.

Além disso, muitas features mostram ter pelo menos dois picos. Logicamente, classes muito distintas de feijão concentram a distribuição delas em pontos específicos. Essas features podem ser especialmente úteis para predição.

In [ ]:
# Tabela de estatísticas descritivas
stats = pd.DataFrame({
    "min": df[FEATURES].min(),
    "max": df[FEATURES].max(),
    "media": df[FEATURES].mean(),
    "desvio": df[FEATURES].std(),
    "assimetria": df[FEATURES].skew(),
})

stats["CV"] = stats["desvio"] / stats["media"]
print("Estatísticas descritivas:")
display(stats)

print(f"\nMaior desvio-padrão: {stats['desvio'].idxmax()} = {stats['desvio'].max():,.2f}")
print(f"Menor desvio-padrão: {stats['desvio'].idxmin()} = {stats['desvio'].min():,.6f}")
print(f"Razão entre maior e menor desvio: {stats['desvio'].max() / stats['desvio'].min():,.0f}")

A tabela de estatísticas mostra uma diferença extrema nas ordens de grandeza entre as features. De fato, muitas features representam razões intrinsecamente limitadas ao intervalo [0, 1], enquanto outras como Area e ConvexArea são representadas por quantidades de pixels em fotos de alta resolução, atingindo a ordem de dezenas de milhares. A padronização se torna essencial.

In [ ]:
# Outliers por feature
q1 = df[FEATURES].quantile(0.25)
q3 = df[FEATURES].quantile(0.75)
iqr = q3 - q1
outls = ((df[FEATURES] < q1 - 1.5 * iqr) | (df[FEATURES] > q3 + 1.5 * iqr)).sum() # Regra IQR
out_tab = pd.DataFrame({"Outliers": outls, "Porcentagem%": (outls / len(df) * 100).round(2)})
print("Outliers por feature:")
display(out_tab.sort_values("Outliers", ascending=False))

#### Relação entre features e alvo

In [ ]:
# Boxplots por classe
box_plts_cls = (
    alt.Chart(df[FEATURES + [TARGET]])
    .mark_boxplot(extent=1.5, size=12, median={"color": "black"}, outliers={"size": 4})
    .encode(
        x = alt.X(f"{TARGET}:N", sort=CLASS_ORDER, title=None,
                axis=alt.Axis(labelAngle=-60, labelFontSize=8)),
        y = alt.Y(alt.repeat("repeat"), type="quantitative"),
        color=class_color(legend=False)
    )
    .properties(width=150, height=130)
    .repeat(repeat=FEATURES, columns=4)
    .resolve_scale(y="independent")
    .properties(title=alt.TitleParams("Distribuição de cada feature por classe",
                                      fontSize=15, anchor="middle"))
)

display(box_plts_cls)

É razoável supor que algumas classes de feijão são muito diferentes em tamanho ou forma. Para testar essa hipótese, fizemos alguns gráficos de dispersão usando apenas pares de features dimensionais OU morfológicas escolhidos por intuição, e observamos como as classes se separam nesses gráficos:

In [ ]:
# Alguns gráficos de dispersão especiais
PAIRS = [
    ("Area", "Perimeter"),
    ("MajorAxisLength", "MinorAxisLength"),
    ("ShapeFactor1", "AspectRatio")
]

brush = alt.selection_interval()

for a, b in PAIRS:
    chart = (
        alt.Chart(df[[a, b, TARGET]])
        .mark_circle(size=14, opacity=0.5, cursor="crosshair")
        .encode(
            x=alt.X(f"{a}:Q", scale=alt.Scale(zero=False)),
            y=alt.Y(f"{b}:Q", scale=alt.Scale(zero=False)),
            color=class_color(),
            tooltip=[
              alt.Tooltip(f"{a}:Q", format=".2f"),
              alt.Tooltip(f"{b}:Q", format=".2f"),
              alt.Tooltip(f"{TARGET}:N", title="Classe")
            ]
        )
        .add_params(brush)
        .interactive()
        .properties(width=480, height=360, title=f"{a} x {b}")
    )
    display(chart)

Eis o que podemor notar em cada gráfico:

*   **Gráfico 1:** Mesmo sem nenhum modelo, a classe Bombay é facilmente identificável nesse gráfico. Além disso, há uma relação quase linear entre Area e Perimeter, mas a relação entre features será explorada mais tarde.
*   **Gráfico 2:** A correlação entre as features também existe, mas é mais fraca que no gráfico anterior. Com menos redundância, a divisão entre as classes é mais clara, porém Bombay continua se destacando.
*   **Gráfico 3:** Essas features foram escolhidas por ter CV alto mesmo sendo morfológicas. O que mais chama atenção é a semelhança com o gráfico 2, mostrando que features dimensionais e morfológicas carregam informação parecida.

Adicionalmente, notamos que nas 4 features relacionadas a tamanho a classe Bombay supera visivelmente as demais. Isso justifica as assimetrias positivas que observamos nessas features anteriormente, já que essa é a classe minoritária.

#### Relações entre features

In [ ]:
# Matriz de correlação de Pearson
corr = df[FEATURES].corr(method="pearson")

pos = {f: i for i, f in enumerate(FEATURES)}
corr_long = (
    corr.rename_axis("feature a").reset_index()
    .melt(id_vars="feature a", var_name="feature b", value_name="r")
)
corr_long = corr_long[corr_long["feature a"].map(pos) > corr_long["feature b"].map(pos)]
corr_long["r_txt"] = corr_long["r"].map(lambda v: f"{v:.2f}")

base = alt.Chart(corr_long).encode(
    x=alt.X("feature b:N", scale=alt.Scale(domain=FEATURES), title=None, axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("feature a:N", scale=alt.Scale(domain=FEATURES), title=None)
)

cells = base.mark_rect(stroke="white", strokeWidth=0.5).encode(
    color=alt.Color(
        "r:Q",
        scale=alt.Scale(scheme="redblue", reverse=True, domain=[-1, 1]),
        legend=alt.Legend(title="Correlação de Pearson", gradientLength=300)
    )
)

values = base.mark_text(fontSize=8).encode(
    text="r_txt:N",
    color=alt.condition("abs(datum.r) > 0.5", alt.value("white"), alt.value("black")),
)

corr_heatmap = (cells + values).properties(
    width=16 * 38, height=16 * 38,
    title="Matriz da correlação de Pearson entre as 16 features")
display(corr_heatmap)

idx_i, idx_j = np.triu_indices(len(FEATURES), k=1)
pairs = pd.DataFrame({
    "feature a": [FEATURES[i] for i in idx_i],
    "feature b": [FEATURES[j] for j in idx_j],
    "r": corr.values[idx_i, idx_j],
})
pairs["|r|"] = pairs["r"].abs()
highest = pairs[pairs["|r|"] > 0.9].sort_values("|r|", ascending=False).reset_index(drop=True)
print(f"De {len(pairs)} pares, {len(highest)} possuem |r| > 0.9:")
display(highest)

Temos um alerta importante: vários pares de features são fortemente correlacionados. É natural, uma vez todas são relacionadas ao tamanho ou forma do grão.

Do ponto de vista de modelos que dependam da distância ou possuem alto custo computacional, isso significa que esse dataset possui uma dimensionalidade desnecessariamente alta. Também surge o problema da multicolinearidade, que afeta métodos lineares.

A implementação de métodos de redução de dimensionalidade, como o PCA, torna se fortemente recomendada.

#### Aprendizado não supervisionado

##### PCA

Como todas as features são numéricas, usamos todas para o PCA. Por padrão, normalizamos previamente.

In [ ]:
FEATURES_CLS = FEATURES

X_cls = df[FEATURES_CLS].to_numpy(dtype=float)
Z_cls = normalize(X_cls)
pca_cls, S_cls, tab_pca_cls, var_exp_cls, var_acum_cls = fit_pca(Z_cls, FEATURES_CLS)
display(tab_pca_cls)

In [ ]:
# Gráfico do cotovelo do PCA
display(pca_elbow_chart(var_exp_cls, var_acum_cls))

O PCA se mostra efetivo ao concentrar mais de 90% da variância em apenas 4 componentes. Tomando essas 4 componentes, resolvemos o problema da alta dimensionalidade perdendo pouca informação.

Quanto as cargas, notamos que PC1 favorece features associadas a tamanho, como Perimeter e MajorAxisLength. Já PC2 é alto para grãos arredondados e de baixa excentricidade. As componentes PC3 e PC4 pesam apenas em componentes com CV baixo, não tendo tanto significado.

##### K-means

In [ ]:
models_cls, wss_cls, tab_wss_cls = sweep_k(S_cls)
display(tab_wss_cls)

In [ ]:
# Gráfico do cotovelo K-means
display(wss_chart(wss_cls))

Para a aplicação de KMeans, construímos o gráfico da WSS e argumentamos que é possível visualizar um cotovelo em K=5, justificando essa escolha de hiperparâmetro. Podemos analisar o perfil dos clusters pela proporção de classes:

In [ ]:
# Proporção de classes por cluster
K_CLS = 5
km_cls = KMeans(k=K_CLS, seed=SEED).fit(S_cls)
cluster_labels = [f"C{i}" for i in range(K_CLS)]

df_cluster = pd.DataFrame({
    "cluster": [cluster_labels[c] for c in km_cls.labels],
    TARGET: df[TARGET].to_numpy(),
})

cluster_class_counts = (
    df_cluster.groupby(["cluster", TARGET])
    .size()
    .reset_index(name="count")
)

cluster_totals = cluster_class_counts.groupby("cluster")["count"].transform("sum")
cluster_class_counts["proportion"] = cluster_class_counts["count"] / cluster_totals

cluster_class_chart = (
    alt.Chart(cluster_class_counts)
    .mark_bar()
    .encode(
        x=alt.X("cluster:N", sort=cluster_labels, title="Cluster"),
        y=alt.Y("proportion:Q", title="Proporção de classes", axis=alt.Axis(format="%")),
        color=class_color(),
        order=alt.Order(f"{TARGET}:N", sort="ascending"),
        tooltip=[
            alt.Tooltip("cluster:N", title="Cluster"),
            alt.Tooltip(f"{TARGET}:N", title="Classe"),
            alt.Tooltip("count:Q", title="Contagem"),
            alt.Tooltip("proportion:Q", title="Proporção", format=".1%")
        ]
    )
    .properties(
        width=480, height=360,
        title="Proporção de classes por cluster (K-means)"
    )
)

display(cluster_class_chart)

A escolha de K=5 gera uma configuração interessante: 3 dos 5 clusters são praticamente dominados por uma única classe. Em especial, 99.8% do cluster C1 está na classe Bombay, que é a classe mais distinta. Por outro lado, Cali e Barbunya permanecem misturadas no cluster C4.

In [ ]:
# Gráfico PCA + centroides
display(projection_chart(S_cls, km_cls, (TARGET, df[TARGET].to_numpy(), class_color())))

Por fim, tentamos visualizar a dispersão das classes usando apenas 2 componentes principais. O resultado se assemelha aos gráficos de dispersão plotados antes.

Adicionando os centroides nessa visualização, percebemos que de fato, as classes se concentram muito bem em 5 clusters. Ainda, como o número de classes é maior do que o de clusters, há fortes sobreposições em alguns pares: Dermason x Sira e Cali x Barbunya. Provavelmente serão os pares mais confundidos nos modelos de classificação.

#### Verificação de plausibilidade dos dados e checagem de leakage

In [ ]:
checks = {}
checks["Area > 0"] = (df["Area"] > 0).all()
checks["Perimeter > 0"] = (df["Perimeter"] > 0).all()
checks["MajorAxisLength >= MinorAxisLength"] = (df["MajorAxisLength"] >= df["MinorAxisLength"]).all()
checks["ConvexArea >= Area"] = (df["ConvexArea"] >= df["Area"]).all()
checks["AspectRatio >= 1"] = (df["AspectRatio"] >= 1).all()

checks["Eccentricity em [0, 1)"] = ((df["Eccentricity"] >= 0) & (df["Eccentricity"] < 1)).all()

# Razões que devem estar em (0, 1].
ratios01 = ["Extent", "Solidity", "Roundness",
             "Compactness", "ShapeFactor3", "ShapeFactor4"]

for f in ratios01:
    checks[f"{f} em (0, 1]"] = ((df[f] > 0) & (df[f] <= 1)).all()

checks_df = pd.DataFrame({"Passou?": pd.Series(checks)})
display(checks_df)

if checks_df["Passou?"].all():
    print("Todas as checagens passaram!")
else:
    failures = (~checks_df["Passou?"]).sum()
    print(f"Total de checagens falhas: {failures}")

Acima, uma bateria de checagens que verifica a plausibilidade lógica dos dados. Não houve problema em nenhuma das checagens. No mais, devemos comentar que nenhuma das features é derivada do tipo do grão, portanto o risco de leakage só existe no caso de uma má divisão dos dados ao treinar os modelos.

# Experimentos de Regressão
<div style="margin: 20px 0;">
  <a href="https://colab.research.google.com/drive/1S62q0hIwksUfK4_qQqOX4nT76rlmcz2V?authuser=1" target="_blank" style="text-decoration: none;">
    <button style="
      background-color: #0284c7;
      color: #ffffff;
      padding: 18px 36px;
      font-size: 18px;
      font-weight: 800;
      letter-spacing: 0.5px;
      border: none;
      border-radius: 10px;
      cursor: pointer;
      display: inline-block;
      box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2);
      transition: background-color 0.2s ease;">
      Experimentos de Regressão
    </button>
  </a>
</div>

# Experimentos de Classificação
<div style="margin: 20px 0;">
  <a href="https://colab.research.google.com/drive/1ZnDAHxPo92M6FfSXsjVQUIng7tPW7HCS?authuser=1" target="_blank" style="text-decoration: none;">
    <button style="
      background-color: #0284c7;
      color: #ffffff;
      padding: 18px 36px;
      font-size: 18px;
      font-weight: 800;
      letter-spacing: 0.5px;
      border: none;
      border-radius: 10px;
      cursor: pointer;
      display: inline-block;
      box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2);
      transition: background-color 0.2s ease;">
      Experimentos de Classificação
    </button>
  </a>
</div>

# Experimentos com Redes Neurais
<div style="margin: 20px 0;">
  <a href="https://colab.research.google.com/drive/1Y8XdEhdWgeRmphhz7jNA_NVDeOpYtp3h?authuser=1" target="_blank" style="text-decoration: none;">
    <button style="
      background-color: #0284c7;
      color: #ffffff;
      padding: 18px 36px;
      font-size: 18px;
      font-weight: 800;
      letter-spacing: 0.5px;
      border: none;
      border-radius: 10px;
      cursor: pointer;
      display: inline-block;
      box-shadow: 0 4px 10px rgba(0, 0, 0, 0.2);
      transition: background-color 0.2s ease;">
      Redes Neurais
    </button>
  </a>
</div>